<a href="https://colab.research.google.com/github/Sachith308/Statistical-Learning-e23308/blob/main/bayesian_inference_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Answers

## Task 1 - Visualizing the 2PL Item Response Model

The Two-Parameter Logistic (2PL) Item Response Theory (IRT) model describes the probability that a user with latent ability $\theta$ correctly answers an item.

The response probability is

$$
P(Y_i=1|\theta)=\frac{1}{1+\exp[-a_i(\theta-b_i)]}
$$

where

- $\theta$ : user's latent ability
- $a_i$ : discrimination parameter
- $b_i$ : difficulty parameter

The discrimination parameter controls how steeply the probability changes around the difficulty point, while the difficulty parameter shifts the curve horizontally.

In [ ]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4, 4, 400)

def irt(theta, a, b):
    return 1/(1 + np.exp(-a*(theta-b)))

fig = go.Figure()

# Same discrimination, different difficulties
a = 1.5

for b in [-1,0,1]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=irt(theta,a,b),
            mode='lines',
            name=f"a={a}, b={b}"
        )
    )

# Different discrimination
fig.add_trace(
    go.Scatter(
        x=theta,
        y=irt(theta,0.7,0),
        mode='lines',
        name="a=0.7, b=0"
    )
)

fig.update_layout(
    title="2PL Item Response Curves",
    xaxis_title="Ability (θ)",
    yaxis_title="P(Correct Response)",
    template="plotly_white"
)

fig.show()

### Interpretation

Increasing the difficulty parameter ($b$) shifts the logistic curve to the right. This means a user requires a higher ability level before achieving the same probability of answering correctly.

Decreasing $b$ shifts the curve to the left, making the item easier.

The discrimination parameter ($a$) controls the steepness of the curve. Larger values of $a$ create a steeper transition from low to high probability, indicating that the item better distinguishes users with abilities near the difficulty level.

## Task 2 - Sequential Likelihood Contribution

For a single response,

$$
P(Y_k=1|\theta)=p_k(\theta)
$$

where

$$
p_k(\theta)=\frac{1}{1+\exp[-a_k(\theta-b_k)]}
$$

The likelihood contribution of one observation is

$$
L(y_k|\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}.
$$

The joint likelihood after observing $k$ responses is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}.
$$

Each new response contributes multiplicatively to the likelihood function.

## Task 3 - Recursive Posterior Update

Bayes' theorem updates the posterior after each new response.

The recursive relationship is

$$
f_k(\theta)
\propto
L(y_k|\theta)
f_{k-1}(\theta)
$$

where

- $f_{k-1}(\theta)$ is the previous posterior,
- $L(y_k|\theta)$ is the likelihood from the newest response.

Expanding the likelihood,

$$
f_k(\theta)
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{k-1}(\theta).
$$

After computing this product numerically, the posterior is normalized so that its total probability equals one.

## Task 4 - Dynamic Shifting of the Posterior

When a user correctly answers a highly difficult question (large $b_k$), the likelihood assigns greater probability to higher ability values.

Consequently,

- the posterior distribution shifts toward larger values of $\theta$,
- the posterior mean increases,
- the MAP estimate also moves to the right.

Therefore, the system becomes more confident that the user has above-average ability.

## Task 5 - Effect of the Discrimination Parameter

The discrimination parameter controls the amount of information contributed by each question.

Large discrimination ($a_k$):

- produces a steeper likelihood,
- creates a sharper posterior,
- reduces posterior variance,
- increases confidence in the ability estimate.

Small discrimination ($a_k$):

- produces a flatter likelihood,
- updates the posterior only slightly,
- results in larger posterior variance,
- provides less information about user ability.

## Task 6 - Numerical Grid Approximation

Since the posterior distribution of the 2PL Item Response Theory (IRT) model does not have a closed-form analytical solution, it is approximated numerically using a fixed grid of ability values.

### Algorithm

1. Create a grid of ability values, $\theta \in [-4,4]$.
2. Evaluate the prior density at every grid point.
3. For each new response:
   - Compute the likelihood at every grid point.
   - Multiply the previous posterior by the likelihood.
   - Normalize the updated posterior using numerical integration:
     
$$
f_k(\theta)
=
\frac{\tilde f_k(\theta)}
{\int \tilde f_k(\theta)d\theta}
$$

where

$$
\tilde f_k(\theta)
=
L(y_k|\theta)f_{k-1}(\theta)
$$

The normalization constant is computed numerically using the trapezoidal rule:

```python
posterior /= np.trapz(posterior, theta_grid)
```

This ensures that the posterior density integrates to one after every update.

## Task 7

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# -------------------------
# Parameters
# -------------------------

np.random.seed(42)

theta_true = 0.75
n = 20

theta_grid = np.linspace(-4,4,500)

prior = norm.pdf(theta_grid)
prior /= np.trapz(prior,theta_grid)

posterior = prior.copy()

posterior_mean = []
posterior_map = []

# Random item parameters

a = np.random.uniform(0.5,2.0,n)
b = np.random.normal(0,1,n)

def logistic(theta,a,b):
    return 1/(1+np.exp(-a*(theta-b)))

# -------------------------
# Sequential Updating
# -------------------------

for k in range(n):

    p_true = logistic(theta_true,a[k],b[k])

    y = np.random.rand() < p_true

    likelihood = logistic(theta_grid,a[k],b[k])

    if y:
        posterior *= likelihood
    else:
        posterior *= (1-likelihood)

    posterior /= np.trapz(posterior,theta_grid)

    mean = np.trapz(theta_grid*posterior,theta_grid)
    MAP = theta_grid[np.argmax(posterior)]

    posterior_mean.append(mean)
    posterior_map.append(MAP)

# -------------------------
# Plot
# -------------------------

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(1,n+1),
    y=posterior_mean,
    mode='lines+markers',
    name='Posterior Mean'
))

fig.add_trace(go.Scatter(
    x=np.arange(1,n+1),
    y=posterior_map,
    mode='lines+markers',
    name='MAP'
))

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True Ability"
)

fig.update_layout(
    title="Sequential Bayesian Ability Estimation",
    xaxis_title="Item Number",
    yaxis_title="Estimated Ability",
    template="plotly_white"
)

fig.show()

/tmp/ipykernel_914/1211098321.py:17: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_914/1211098321.py:49: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_914/1211098321.py:51: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



## Analysis

Initially, the posterior mean and MAP estimates fluctuate because only a few responses are available, resulting in high uncertainty about the user's ability.

As additional responses are observed, the posterior distribution becomes increasingly concentrated around the true ability. Consequently, both the Posterior Mean and the MAP estimate gradually converge toward the true latent ability value of $\theta = 0.75$.

The decreasing difference between the estimators and the true ability indicates that the platform is accumulating evidence and becoming more confident in its estimate. Although small fluctuations may still occur due to random item difficulties and discrimination values, the overall trend demonstrates successful Bayesian learning through sequential updating.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

## Answer

## Task 1 - Structural Probability and Properties

The click-through rate (CTR) is modeled as a probability parameter

$$
\theta \in [0,1]
$$

with the prior distribution

$$
\Theta \sim \text{Beta}(\alpha,\beta).
$$

The Beta probability density function is

$$
f(\theta)=
\frac{\theta^{\alpha-1}(1-\theta)^{\beta-1}}
{B(\alpha,\beta)},
\qquad 0\le\theta\le1
$$

where

- $\alpha$ represents prior evidence for clicks.
- $\beta$ represents prior evidence for non-clicks.

Three different parameter settings are considered:

- Beta(1,1): Uniform prior (no prior preference).
- Beta(2,8): Right-skewed distribution favouring low CTR.
- Beta(8,2): Left-skewed distribution favouring high CTR.

Changing the balance between $\alpha$ and $\beta$ shifts the center of the distribution. Larger $\alpha$ shifts the density toward higher click probabilities, while larger $\beta$ shifts it toward lower click probabilities.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0,1,500)

fig = go.Figure()

params = [(1,1),(2,8),(8,2)]

for a,b in params:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=beta.pdf(theta,a,b),
            mode="lines",
            name=f"Beta({a},{b})"
        )
    )

fig.update_layout(
    title="Beta Prior Distributions",
    xaxis_title="CTR (θ)",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

## Task 2 - Sequential Likelihood

Each user interaction follows a Bernoulli distribution,

$$
Y_k|\theta \sim \text{Bernoulli}(\theta).
$$

The likelihood contribution of one observation is

$$
L(y_k|\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

After observing $k$ impressions, the joint likelihood becomes

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
$$

Each click increases the exponent of $\theta$, while each non-click increases the exponent of $(1-\theta)$.

## Task 3 - Beta-Binomial Conjugacy

Assume the prior distribution is

$$
\Theta \sim \text{Beta}(\alpha_{k-1},\beta_{k-1}).
$$

After observing one new response $y_k$, Bayes' theorem gives

$$
f_k(\theta)
\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
f_{k-1}(\theta).
$$

Since the Beta distribution is conjugate to the Bernoulli likelihood, the posterior is also Beta:

$$
\Theta|Y^{(k)}
\sim
\text{Beta}(\alpha_k,\beta_k)
$$

where

$$
\alpha_k=\alpha_{k-1}+y_k
$$

$$
\beta_k=\beta_{k-1}+1-y_k
$$

The posterior mean is

$$
E[\Theta|Y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$

Therefore, Bayesian updating only requires adding the new click or non-click count to the previous parameters.

## Task 4 - Dynamic Shifting

When a click is observed ($y_k=1$),

$$
\alpha_k=\alpha_{k-1}+1
$$

while

$$
\beta_k=\beta_{k-1}.
$$

This shifts the Beta distribution toward larger CTR values.

When a non-click is observed ($y_k=0$),

$$
\beta_k=\beta_{k-1}+1
$$

while

$$
\alpha_k=\alpha_{k-1}.
$$

The distribution shifts toward smaller CTR values.

Unlike the 2PL Item Response Theory model, these updates have exact closed-form solutions because the Beta prior is conjugate to the Bernoulli likelihood. Therefore, no numerical grid approximation is required.

## Task 5 - Running Point Estimators

The running Posterior Mean is

$$
\hat{\theta}_{Bayes}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$

The Maximum A Posteriori (MAP) estimate is

$$
\hat{\theta}_{MAP}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2},
\qquad
\alpha_k>1,\;
\beta_k>1.
$$

The Posterior Mean minimizes expected squared error, while the MAP estimate corresponds to the mode of the posterior distribution.

## Task 6

In [ ]:
import numpy as np
import plotly.graph_objects as go

# -----------------------------
# Parameters
# -----------------------------
np.random.seed(42)

theta_true = 0.35
n = 100

alpha = 1
beta = 1

posterior_mean = []
posterior_map = []

# -----------------------------
# Sequential Bayesian Updating
# -----------------------------
for k in range(n):

    # Simulate a click (1) or non-click (0)
    y = np.random.rand() < theta_true

    # Update Beta parameters
    if y:
        alpha += 1
    else:
        beta += 1

    # Posterior Mean
    mean = alpha / (alpha + beta)

    # MAP Estimate
    if alpha > 1 and beta > 1:
        MAP = (alpha - 1) / (alpha + beta - 2)
    else:
        MAP = mean

    posterior_mean.append(mean)
    posterior_map.append(MAP)

# -----------------------------
# Plot Results
# -----------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=np.arange(1, n + 1),
        y=posterior_mean,
        mode="lines",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=np.arange(1, n + 1),
        y=posterior_map,
        mode="lines",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR"
)

fig.update_layout(
    title="Sequential Bayesian Estimation of Click-Through Rate",
    xaxis_title="Number of Impressions",
    yaxis_title="Estimated CTR",
    template="plotly_white"
)

fig.show()

## Analysis

At the beginning of the simulation, the Posterior Mean and MAP estimates fluctuate noticeably because only a few user interactions have been observed. With limited evidence, the prior distribution has a relatively strong influence on the estimated click-through rate.

As more impressions are collected, both estimators gradually stabilize and converge toward the true click-through rate of **0.35**. The gap between the estimates and the true value decreases because each additional observation contributes more information, reducing uncertainty in the posterior distribution.

Compared with non-conjugate Bayesian models, the Beta-Binomial model allows efficient sequential updating using simple arithmetic operations on the Beta parameters. This makes it computationally efficient for real-time applications such as online advertising, recommendation systems, and A/B testing, where user responses arrive continuously.

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Answers

## Task 1 - Prior Belief Boundaries

The remaining structural stiffness efficiency factor is represented by

$$
\theta \in (0,1]
$$

where:

- $\theta = 1$ represents a completely healthy structure.
- Smaller values of $\theta$ indicate increasing structural degradation.

The prior belief is modeled as

$$
\Theta \sim \text{Beta}(8,1.5)
$$

whose probability density function is

$$
f(\theta)=
\frac{\theta^{8-1}(1-\theta)^{1.5-1}}
{B(8,1.5)},
\qquad 0<\theta<1.
$$

The expected value of a Beta distribution is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}
=
\frac{8}{8+1.5}
=
0.842.
$$

This prior reflects an engineering assumption that newly installed structures are most likely healthy before any measurements are collected, while still allowing uncertainty due to manufacturing tolerances and environmental conditions.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0.01,1,500)

prior = beta.pdf(theta,8,1.5)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta,
        y=prior,
        mode='lines',
        name='Beta(8,1.5)'
    )
)

fig.update_layout(
    title="Prior Distribution of Structural Stiffness",
    xaxis_title="Remaining Stiffness (θ)",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

## Task 2 - Structural Likelihood

The observed sensor measurement follows

$$
y_k
=
\theta K_{nominal}e^{\varepsilon_k}
$$

where

$$
\varepsilon_k
\sim
N(0,\sigma^2).
$$

Therefore,

$$
\ln(y_k)
=
\ln(\theta K_{nominal})
+
\varepsilon_k.
$$

The likelihood of one observation is

$$
L(y_k|\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-\frac{
(\ln y_k-\ln(\theta K_{nominal}))^2
}
{2\sigma^2}
\right].
$$

The joint likelihood after observing multiple sensor readings is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
L(y_i|\theta).
$$

## Task 3 - Non-Conjugate Posterior Update

The prior distribution is Beta, while the likelihood is Log-Normal.

Since these distributions are not conjugate, there is no analytical closed-form expression for the posterior distribution.

Instead, Bayes' theorem is applied numerically:

$$
f_k(\theta)
\propto
L(y_k|\theta)
f_{k-1}(\theta).
$$

After computing the product, the posterior distribution is normalized numerically over the bounded interval

$$
\theta\in(0,1].
$$

Therefore, every new sensor measurement updates the posterior distribution through numerical grid approximation rather than simple parameter updates.

## Task 4 - Running Point Estimators

Since no closed-form posterior exists, the estimators are computed numerically.

The Posterior Mean is

$$
\hat{\theta}_{Bayes}
=
\frac{
\int_{0.01}^{1}
\theta
f(\theta|y^{(k)})
d\theta
}{
\int_{0.01}^{1}
f(\theta|y^{(k)})
d\theta
}.
$$

The MAP estimate is

$$
\hat{\theta}_{MAP}
=
\underset{\theta}{\operatorname{argmax}}
\;
f(\theta|y^{(k)}).
$$

The Posterior Mean minimizes the expected squared error, whereas the MAP estimate corresponds to the most probable stiffness value according to the posterior distribution.

## Task 5 - Numerical Grid Approximation

The posterior distribution is maintained numerically on a fixed grid of stiffness values.

Algorithm:

1. Create a grid of θ values between 0.01 and 1.00.
2. Compute the Beta prior density on the grid.
3. For each sensor measurement:
   - Evaluate the Log-Normal likelihood for every θ value.
   - Multiply the previous posterior by the likelihood.
   - Normalize the posterior using the trapezoidal rule

$$
f_k(\theta)
=
\frac{
\tilde f_k(\theta)
}{
\int_{0.01}^{1}
\tilde f_k(\theta)
d\theta
}
$$

where

$$
\tilde f_k(\theta)
=
L(y_k|\theta)
f_{k-1}(\theta).
$$

In Python, normalization is performed using

```python
posterior /= np.trapz(posterior, theta_grid)
```

The bounded grid ensures that all estimates remain within the physically meaningful interval of 0.01 to 1.00.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# --------------------------------------------------
# Parameters
# --------------------------------------------------

np.random.seed(42)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n = 15

# Grid
theta_grid = np.linspace(0.01,1.0,500)

# Prior Beta(8,1.5)
posterior = beta.pdf(theta_grid,8,1.5)
posterior /= np.trapz(posterior,theta_grid)

posterior_mean = []
posterior_map = []

saved_posteriors = {0:posterior.copy()}

# --------------------------------------------------
# Sequential Updating
# --------------------------------------------------

for k in range(1,n+1):

    # Simulate measurement
    epsilon = np.random.normal(0,sigma)
    y = theta_true * K_nominal * np.exp(epsilon)

    # Lognormal likelihood
    likelihood = (
        1/(y*sigma*np.sqrt(2*np.pi))
        * np.exp(
            -(
                np.log(y)-np.log(theta_grid*K_nominal)
            )**2
            /(2*sigma**2)
        )
    )

    posterior *= likelihood
    posterior /= np.trapz(posterior,theta_grid)

    mean = np.trapz(theta_grid*posterior,theta_grid)
    MAP = theta_grid[np.argmax(posterior)]

    posterior_mean.append(mean)
    posterior_map.append(MAP)

    if k in [1,2,5,10,15]:
        saved_posteriors[k] = posterior.copy()

# --------------------------------------------------
# Posterior Density Evolution
# --------------------------------------------------

fig = go.Figure()

for step in [0,1,2,5,10,15]:
    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=saved_posteriors[step],
            mode="lines",
            name=f"Step {step}"
        )
    )

fig.update_layout(
    title="Evolution of Posterior Density",
    xaxis_title="Remaining Stiffness (θ)",
    yaxis_title="Posterior Density",
    template="plotly_white"
)

fig.show()

# --------------------------------------------------
# Estimator Convergence
# --------------------------------------------------

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=np.arange(1,n+1),
        y=posterior_mean,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig2.add_trace(
    go.Scatter(
        x=np.arange(1,n+1),
        y=posterior_map,
        mode="lines+markers",
        name="MAP"
    )
)

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True Stiffness"
)

fig2.update_layout(
    title="Sequential Estimation of Structural Stiffness",
    xaxis_title="Inspection Step",
    yaxis_title="Estimated θ",
    template="plotly_white"
)

fig2.show()

/tmp/ipykernel_914/2479427765.py:21: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_914/2479427765.py:50: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_914/2479427765.py:52: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



## Analysis

Initially, the posterior distribution is concentrated near higher stiffness values because the Beta(8,1.5) prior assumes that the structure is likely to be healthy before any measurements are collected.

As additional sensor measurements are incorporated, the posterior gradually shifts toward the true stiffness value of **θ = 0.68**. The posterior mean and MAP estimates converge steadily to this value, indicating that the influence of the prior decreases while the observed data become increasingly dominant.

The posterior density also becomes progressively narrower with each update, reflecting a reduction in uncertainty. This increased concentration indicates greater confidence in the estimated structural condition.

In practice, only a few sensor readings are needed before the posterior moves away from the initially optimistic prior. As more observations accumulate, the Bayesian estimator accurately identifies the degraded stiffness level. The narrowing of the posterior distribution implies improved confidence in structural health assessment, enabling engineers to make more reliable maintenance and safety decisions.

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

## Answers

## Task 1 - Deriving the Marginal Density

Let the prior probability of belonging to cluster $k$ be

$$
P(C_i=k)=\phi_k,
$$

where

$$
\sum_{k=1}^{K}\phi_k=1.
$$

Conditioned on cluster membership,

$$
X_i|C_i=k
\sim
\mathcal N(\mu_k,\Sigma_k).
$$

Using the law of total probability,

$$
p(x_i)
=
\sum_{k=1}^{K}
P(C_i=k)
p(x_i|C_i=k).
$$

Substituting the Gaussian density,

$$
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal N(x_i|\mu_k,\Sigma_k).
$$

This is called a **Gaussian Mixture Density** because the overall probability density is a weighted sum of several Gaussian distributions. Each Gaussian represents one cluster, while the mixture weights represent the probability that an observation belongs to each cluster.

## Task 2 - Posterior Cluster Probability

Using Bayes' theorem,

$$
P(C_i=k|X_i=x_i)
=
\frac{
P(X_i=x_i|C_i=k)P(C_i=k)
}{
\sum_{j=1}^{K}
P(X_i=x_i|C_i=j)P(C_i=j)
}.
$$

Substituting the Gaussian likelihood,

$$
P(C_i=k|X_i=x_i)
=
\frac{
\phi_k
\mathcal N(x_i|\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal N(x_i|\mu_j,\Sigma_j)
}.
$$

This posterior probability is called the **responsibility**

$$
\gamma_{ik}
=
P(C_i=k|X_i=x_i).
$$

The responsibility represents the probability that observation $x_i$ belongs to cluster $k$ after considering both the prior probability and the observed data.

## Task 3 - One-Hot Encoding of the Latent Variable

Define the latent vector

$$
Z_i=
(Z_{i1},Z_{i2},...,Z_{iK}),
$$

where

$$
Z_{ik}
=
\begin{cases}
1,&C_i=k\\
0,&\text{otherwise}
\end{cases}
$$

The conditional expectation is

$$
E[Z_{ik}|X_i=x_i]
=
P(C_i=k|X_i=x_i)
=
\gamma_{ik}.
$$

Hence,

$$
E[Z_i|X_i=x_i]
=
(\gamma_{i1},\gamma_{i2},...,\gamma_{iK})^T.
$$

Therefore, the responsibility vector is the expected value of the latent one-hot cluster indicator and represents the soft assignment of each observation across all clusters.

## Task 4 - Soft and Hard Clustering

The soft assignment vector

$$
(\gamma_{i1},\gamma_{i2},...,\gamma_{iK})
$$

contains the probability that observation $x_i$ belongs to every cluster.

A hard assignment is obtained by selecting the cluster with the largest posterior probability:

$$
\hat C_i
=
\operatorname*{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$

Soft clustering allows an observation to partially belong to multiple clusters, reflecting uncertainty in the assignment. In contrast, hard clustering assigns each observation to only one cluster based on the highest posterior probability.

## Task 5 - Conditional Expectation of the Observation

For a Gaussian distribution,

$$
E[X_i|C_i=k]
=
\mu_k.
$$

Therefore, the cluster mean represents the center of cluster $k$.

Two important conditional expectations are

$$
E[Z_i|X_i=x_i]
=
(\gamma_{i1},...,\gamma_{iK}),
$$

and

$$
E[X_i|C_i=k]
=
\mu_k.
$$

The first expectation provides the posterior probability of cluster membership for an observed data point, while the second gives the expected location (center) of a specific cluster.

Thus, one describes **which cluster an observation belongs to**, whereas the other describes **where the cluster is located** in the feature space.

## Task 6 - Complete Data Likelihood

If the latent cluster labels are known, the complete-data likelihood is

$$
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathcal{N}(x_i|\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

Taking the natural logarithm gives the complete-data log-likelihood

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log\mathcal N(x_i|\mu_k,\Sigma_k)
\right].
$$

If the values of $z_{ik}$ were known, every observation would belong to one cluster, allowing the parameters of each cluster to be estimated independently. Therefore, maximizing the complete-data log-likelihood would be straightforward.

## Task 7 - Expectation-Maximization (EM) Interpretation

Since the true cluster memberships are unknown, the EM algorithm replaces each latent variable with its conditional expectation.

The E-step computes

$$
z_{ik}
\rightarrow
E[Z_{ik}|X_i=x_i]
=
\gamma_{ik}.
$$

The expected complete-data log-likelihood becomes

$$
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log\mathcal N(x_i|\mu_k,\Sigma_k)
\right].
$$

The E-step updates the probability that each observation belongs to every cluster. These probabilities (responsibilities) are then used as weights in the M-step to update the model parameters.

## Task 8 - Parameter Updates

The effective number of observations assigned to cluster $k$ is

$$
N_k
=
\sum_{i=1}^{n}
\gamma_{ik}.
$$

The updated mixture weight is

$$
\phi_k^{new}
=
\frac{N_k}{n}.
$$

The updated cluster mean is

$$
\mu_k^{new}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i.
$$

The updated covariance matrix is

$$
\Sigma_k^{new}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k)
(x_i-\mu_k)^T.
$$

The responsibility acts as a fractional membership weight. Observations with larger responsibilities contribute more strongly to the estimation of the corresponding cluster parameters.

## Task 9 - Interpretation

Gaussian Mixture Model (GMM) clustering is a probabilistic clustering method based on repeated conditional updating. Initially, each cluster has a prior probability represented by the mixture weight. For every observation, Bayes' theorem computes the posterior probability (responsibility) that the observation belongs to each cluster. These responsibilities are then used as weighted memberships to update the cluster parameters during the M-step. This alternating process continues until convergence, allowing both the cluster assignments and model parameters to improve iteratively. Unlike hard clustering methods such as K-Means, GMM naturally represents uncertainty by allowing observations to belong to multiple clusters with different probabilities.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

# ---------------------------------------------------
# Load Dataset
# ---------------------------------------------------

# Replace with your dataset path after uploading 'CC GENERAL.csv' to your Colab environment
df = pd.read_csv("CC GENERAL.csv")

# Select two continuous features
X = df[['PURCHASES','CREDIT_LIMIT']].fillna(df.mean(numeric_only=True))

# ---------------------------------------------------
# Standardization
# ---------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------------------------------------------------
# Train-Test Split
# ---------------------------------------------------

X_train, X_test = train_test_split(
    X_scaled,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------
# Fit GMM
# ---------------------------------------------------

gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    random_state=42
)

gmm.fit(X_train)

print("Converged:", gmm.converged_)
print("Iterations:", gmm.n_iter_)

# ---------------------------------------------------
# Test Log-Likelihood
# ---------------------------------------------------

score = gmm.score(X_test)

print("Average Test Log-Likelihood:", score)

# ---------------------------------------------------
# Training Assignments
# ---------------------------------------------------

train_labels = gmm.predict(X_train)

fig1 = px.scatter(
    x=X_train[:,0],
    y=X_train[:,1],
    color=train_labels.astype(str),
    title="Training Cluster Assignments",
    labels={
        "x":"PURCHASES (scaled)",
        "y":"CREDIT_LIMIT (scaled)"
    }
)

fig1.show()

# ---------------------------------------------------
# Test Assignments
# ---------------------------------------------------

test_labels = gmm.predict(X_test)

fig2 = px.scatter(
    x=X_test[:,0],
    y=X_test[:,1],
    color=test_labels.astype(str),
    title="Test Cluster Assignments",
    labels={
        "x":"PURCHASES (scaled)",
        "y":"CREDIT_LIMIT (scaled)"
    }
)

fig2.show()

# ---------------------------------------------------
# Density Heatmap
# ---------------------------------------------------

fig3 = px.density_heatmap(
    x=X_train[:,0],
    y=X_train[:,1],
    nbinsx=40,
    nbinsy=40,
    title="Training Data Density"
)

fig3.show()

Converged: True
Iterations: 19
Average Test Log-Likelihood: -1.582885574788247


## Analysis

The Gaussian Mixture Model successfully partitions the dataset into three probabilistic clusters using the Expectation-Maximization (EM) algorithm. The model converges after a finite number of iterations, indicating stable parameter estimation.

The average test log-likelihood measures how well the learned mixture distribution generalizes to unseen observations. Higher values indicate a better fit to the underlying data distribution.

The density heatmap reveals regions where customer observations are concentrated, suggesting potential multimodal behavior. The training and test cluster assignment plots show how observations are grouped based on their posterior probabilities.

Unlike hard clustering methods, the GMM models uncertainty by assigning probabilities of belonging to each cluster. This probabilistic approach enables smoother decision boundaries and provides more informative clustering results for complex datasets.